# Proof of Concept (PoC) - LLM XAI Evaluation Framework

Niniejszy notebook demonstruje działanie modułowego ramy badawczej (w `src/`) do oceny metod interpretowalności modeli językowych (**SHAP** oraz **Occlusion/LOO**).

### Zakres PoC:
1. **Model:** `microsoft/phi-2` (Causal LM) w precyzji `bfloat16` na GPU.
2. **Obliczenie atrybucji:** SHAP vs Occlusion (Leave-One-Out).
3. **Ocena Stabilności (Stability):** Korelacja Spearmana i Jaccard Similarity po perturbacji tekstu.
4. **Ocena Wierności (Faithfulness):** Spadek prawdopodobieństwa (*Probability Drop*) po usunięciu najważniejszych tokenów.
5. **Koszt Obliczeniowy:** Porównanie czasu generowania wyjaśnień.

In [ ]:
import sys
import os
import json
import pandas as pd
from datasets import load_dataset

# Dodajemy katalog src/ do ścieżki Pythona
sys.path.append(os.path.abspath("../src"))

from models import SentimentCausalLM
from explainers import SHAPExplainer, OcclusionExplainer
from perturbations import TextPerturber
from metrics import stability_spearman, stability_topk_jaccard, faithfulness_prob_drop

In [ ]:
# 1. Inicjalizacja modelu Causal LM na GPU
model = SentimentCausalLM(model_name="microsoft/phi-2")

In [ ]:
# 2. Pobranie przykładowych zdań ze zbioru SST-2 (z lokalnego pliku lub HuggingFace)
local_data_path = "../data/sst2_validation.json"
if os.path.exists(local_data_path):
    with open(local_data_path, "r", encoding="utf-8") as f:
        data_json = json.load(f)[:5]
    samples = [item["sentence"] for item in data_json]
    labels = [item["label"] for item in data_json]
else:
    dataset = load_dataset("stanfordnlp/sst2", split="validation[:5]")
    samples = dataset["sentence"]
    labels = dataset["label"]

print(f"Pobrano {len(samples)} próbek do testu PoC.")
for i, s in enumerate(samples):
    print(f"[{i}] Label={labels[i]}: {s}")

In [ ]:
# 3. Inicjalizacja klas Wyjaśniających (Explainers)
shap_explainer = SHAPExplainer(model, max_evals=100)
occlusion_explainer = OcclusionExplainer(model)

In [ ]:
# 4. Przeprowadzenie eksperymentu PoC dla próbek SST-2
results = []

for idx, text in enumerate(samples):
    true_label = labels[idx]
    pred_class, probs = model.predict_single(text)
    
    # Generowanie wyjaśnień
    res_shap = shap_explainer.explain(text, target_class=pred_class)
    res_occ = occlusion_explainer.explain(text, target_class=pred_class)
    
    # Perturbacja tekstu (dodanie neutralnego prefixu 'Overall, ')
    pert_text = TextPerturber.add_prefix(text, "Overall, ")
    res_shap_pert = shap_explainer.explain(pert_text, target_class=pred_class)
    res_occ_pert = occlusion_explainer.explain(pert_text, target_class=pred_class)
    
    # Metryki Stabilności (Stability)
    shap_spearman = stability_spearman(res_shap, res_shap_pert)
    occ_spearman = stability_spearman(res_occ, res_occ_pert)
    
    shap_jaccard = stability_topk_jaccard(res_shap, res_shap_pert, k=3)
    occ_jaccard = stability_topk_jaccard(res_occ, res_occ_pert, k=3)
    
    # Metryki Wierności (Faithfulness - Prob Drop po usunięciu Top-2 tokenów)
    shap_faith = faithfulness_prob_drop(model, text, res_shap, k=2)
    occ_faith = faithfulness_prob_drop(model, text, res_occ, k=2)
    
    results.append({
        "Sample_ID": idx,
        "Text": text[:35] + "...",
        "True_Label": true_label,
        "Pred_Label": pred_class,
        "P_target": f"{probs[pred_class]:.3f}",
        "SHAP_Time(s)": f"{res_shap.execution_time:.2f}",
        "Occ_Time(s)": f"{res_occ.execution_time:.2f}",
        "SHAP_Spearman": f"{shap_spearman:.3f}",
        "Occ_Spearman": f"{occ_spearman:.3f}",
        "SHAP_Jaccard": f"{shap_jaccard:.3f}",
        "Occ_Jaccard": f"{occ_jaccard:.3f}",
        "SHAP_ProbDrop": f"{shap_faith:.3f}",
        "Occ_ProbDrop": f"{occ_faith:.3f}"
    })

df_results = pd.DataFrame(results)
df_results

In [ ]:
# 5. Podsumowanie wyników PoC
print("=== PODSUMOWANIE METRYK POC ===")
print(f"Sredni czas SHAP: {df_results['SHAP_Time(s)'].astype(float).mean():.2f}s | Occlusion: {df_results['Occ_Time(s)'].astype(float).mean():.2f}s")
print(f"Srednia Stabilnosc Spearman -> SHAP: {df_results['SHAP_Spearman'].astype(float).mean():.3f} | Occlusion: {df_results['Occ_Spearman'].astype(float).mean():.3f}")
print(f"Srednia Wiernosc (Prob Drop) -> SHAP: {df_results['SHAP_ProbDrop'].astype(float).mean():.3f} | Occlusion: {df_results['Occ_ProbDrop'].astype(float).mean():.3f}")